In [ ]:
!pip install roboflow ultralytics

from roboflow import Roboflow
rf = Roboflow(api_key="gG6JYYxaCfcQ7M5NNSfn")
project = rf.workspace("tom-rowland-03ams").project("taco-wikc7")
version = project.version(3)
dataset = version.download("yolov8-obb")


Iniciar Treinamento

In [ ]:
import yaml
import os

# Path to the data.yaml file
data_yaml_path = f"{dataset.location}/data.yaml"

# Read the data.yaml file
with open(data_yaml_path, 'r') as f:
    data_yaml_content = yaml.safe_load(f)

# Update the 'path' in data.yaml to the absolute path of the dataset
data_yaml_content['path'] = os.path.abspath(dataset.location)

# Save the updated data.yaml file
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml_content, f)

from ultralytics import YOLO

model = YOLO('yolov8n-obb.pt')

# Treinar o modelo com os dados do TACO
results = model.train(
    data=data_yaml_path,  # Use the path to the updated data.yaml
    epochs=50,
    imgsz=640,
    batch=16,
    name='modelo_lixeira_inteligente'
)

Testando Modelo e Gerando JSON para a aplicação web

In [ ]:
from ultralytics import YOLO
from google.colab import files
import IPython.display as display
import os
import json
from datetime import datetime, timezone

caminho_do_modelo = '/content/runs/obb/modelo_lixeira_inteligente3/weights/best.pt'
modelo_treinado = YOLO(caminho_do_modelo)

print("Faça o upload de uma imagem para testar a detecção na lixeira:")
uploaded = files.upload()

for nome_arquivo in uploaded.keys():
    caminho_imagem = f"/content/{nome_arquivo}"
    print(f"\nAnalisando a imagem: {nome_arquivo}...")

    resultados = modelo_treinado.predict(source=caminho_imagem, save=True)

    payload = {
        "lixeira_id": "smart_bin_01",
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "detections": []
    }

    # Process detections for JSON payload
    for resultado in resultados:
        if resultado.obb is not None and len(resultado.obb) > 0: # Check if there are actual detections
            for box in resultado.obb:
                indice_classe = int(box.cls[0].item())
                nome_da_classe = resultado.names[indice_classe]
                certeza = box.conf[0].item()

                x, y, w, h, r = box.xywhr[0].tolist()

                payload["detections"].append({
                    "class_name": nome_da_classe,
                    "confidence": round(certeza, 4),
                    "bounding_box": {
                        "x_center": round(x, 2),
                        "y_center": round(y, 2),
                        "width": round(w, 2),
                        "height": round(h, 2),
                        "angle": round(r, 4)
                    }
                })

            # Print detection details only if detections exist
            if len(payload["detections"]) > 0:
                print(f"\n--- RESÍDUO DETECTADO ---")
                for det in payload["detections"]:
                    print(f"Tipo: {det['class_name'].upper()}")
                    print(f"Certeza: {det['confidence'] * 100:.2f}%")
        else:
            print("\n--- NENHUM RESÍDUO DETECTADO ---")


    print("\n--- JSON GERADO PARA A API ---")
    print(json.dumps(payload, indent=2))

    # Display the image
    pasta_salvamento = resultados[0].save_dir
    caminho_imagem_salva = os.path.join(pasta_salvamento, nome_arquivo) # This is the expected path

    if os.path.exists(caminho_imagem_salva):
        print(f"\n--- IMAGEM COM DETECÇÕES SALVA ---")
        display.display(display.Image(filename=caminho_imagem_salva))
    else:
        print(f"\n--- IMAGEM ORIGINAL (NÃO ANOTADA) EXIBIDA ---")
        print(f"Nota: Nenhuma imagem anotada foi salva em '{caminho_imagem_salva}', provavelmente porque nenhum objeto foi detectado ou devido ao comportamento de salvamento do modelo.")
        display.display(display.Image(filename=caminho_imagem)) # Fallback to original image